In [ ]:
import os
from copy import deepcopy
import numpy as np
import random
import torch
import umap
random_seed = 2025
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index

from utils import calc_logit_norm

In [ ]:
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

In [ ]:
def add_gaussian_noise(X, severity=5):
    scale = [.08, .12, 0.18, 0.26, 0.38][severity - 1]*5
    # Add Gaussian noise to the data
    generator = torch.Generator().manual_seed(random_seed)
    noise = torch.normal(size=X.shape, std=scale, mean=0.0, generator=generator)
    noisy_X = X + noise
    return noisy_X

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
from sklearn.model_selection import train_test_split



class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, num_classes=3):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x, return_features=False):
        x = F.relu(self.bn1(self.fc1(x)))
        features = F.relu(self.bn2(self.fc2(x)))
        logits = self.fc3(features)
        if return_features:
            return logits, features
        return logits


class MLPGN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, num_classes=3, num_groups=4):
        super(MLPGN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.gn1 = nn.GroupNorm(num_groups, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.gn2 = nn.GroupNorm(num_groups, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_features=False):
        x = self.fc1(x)
        x = self.gn1(x.unsqueeze(2)).squeeze(2)
        x = F.relu(x)

        x = self.fc2(x)
        x = self.gn2(x.unsqueeze(2)).squeeze(2)
        features = F.relu(x)

        if return_features:
            return self.fc3(features), features
        
        return self.fc3(features)
            

In [ ]:
class PointDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

def validate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            # Get predictions from the maximum value
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy

In [ ]:
# 1. Create a synthetic dataset with 3 classes
n_samples = 2000
num_classes = 3
centers = [(0, 3), (-3, -3), (3, -3)]
colors = ['r', 'g', 'b']

X, y = make_blobs(n_samples=n_samples, centers=centers, n_features=2, random_state=random_seed)

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=random_seed)
x_train, x_val = torch.tensor(x_train, dtype=torch.float32), torch.tensor(x_val, dtype=torch.float32)
y_train, y_val = torch.tensor(y_train, dtype=torch.long), torch.tensor(y_val, dtype=torch.long)


train_dataloader = torch.utils.data.DataLoader(PointDataset(x_train, y_train), 
                                                  batch_size=64, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(PointDataset(x_val, y_val),
                                                batch_size=64, shuffle=False)

In [ ]:

# model = MLP(num_classes=num_classes)
# # model = MLPGN(num_classes=num_classes)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.01)

# # 3. Train the model on the clean data
# train_losses = []
# val_losses = []
# n_epochs = 100
# model.train()

# for epoch in range(n_epochs):
#     model.train()
#     for x_data, y_data in train_dataloader:
#         optimizer.zero_grad()
#         outputs = model(x_data)
#         loss = criterion(outputs, y_data)
#         loss.backward()
#         optimizer.step()
        
#         train_loss = loss.item()

#     train_losses.append(train_loss)
        
#     # Validation
#     model.eval()
#     with torch.no_grad():
#         val_outputs = model(x_val)
#         val_loss = criterion(val_outputs, y_val).item()
#         val_losses.append(val_loss)
        
#     if (epoch+1) % 10 == 0:
#         print(f"Epoch [{epoch+1}/{n_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# plt.figure(figsize=(6,3))
# plt.plot(train_losses, label='Train Loss')
# plt.plot(val_losses, label='Validation Loss')
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.title("Training and Validation Loss")
# plt.legend()
# plt.show()

# torch.save(model, '/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/point_mlp.pth')

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/point_mlp.pth')
model = model.to('cuda')

In [ ]:
# 6. Visualize the features before and after adaptation using PCA
def plot_features(model, features, labels, title, pca=None):
    features_np = features.detach().cpu().numpy()

    if pca is None:
        pca = PCA(n_components=2, random_state=random_seed)
        features_2d = pca.fit_transform(features_np)
    else:
        features_2d = pca.transform(features_np)
    plt.figure(figsize=(6,5))

    # # create a grid for decision boundaries
    # x_min, x_max = features_2d[:, 0].min() - 1, features_2d[:, 0].max() + 1
    # y_min, y_max = features_2d[:, 1].min() - 1, features_2d[:, 1].max() + 1
    # xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

    # model.eval()
    # with torch.no_grad():
    #     Z = model(torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32))
    #     Z = torch.argmax(Z, dim=1).reshape(xx.shape)

    # plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.Spectral)

    for i in range(num_classes):
        plt.scatter(features_2d[labels==i, 0], features_2d[labels==i, 1], c=colors[i], label=f'Class {i}', alpha=0.7)
    plt.legend()
    plt.title(title)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()

    return pca

def visualize_features(features, labels, title="Umap Visualization of CNN Feature Outputs", reducer=None):
    
    if reducer is None:
        reducer = umap.UMAP(random_state=42)
        embedding = reducer.fit_transform(features)
    else:
        embedding = reducer.transform(features)

    # Plot the UMAP results
    plt.figure(figsize=(8, 6))

    for i in range(num_classes):
        plt.scatter(embedding[labels==i, 0], embedding[labels==i, 1], c=colors[i], label=f'Class {i}', alpha=0.7)
    plt.legend()
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.title(title)
    plt.show()
    return reducer

def extract_features(model, data_loader):
    # Create a list to store the features and labels
    fc_features = []
    fc_labels = []

    # Define a forward hook to capture the output of fc1 (the hidden fully connected layer)
    def hook(module, input, output):
        # Append the output features and corresponding labels
        fc_features.append(output.detach().cpu())

    # Register the hook on the first fully connected layer (bn2)
    hook_handle = model.bn2.register_forward_hook(hook)

    # Make sure the model is in evaluation mode
    model.eval()

    # Iterate over the test loader to extract features
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to('cuda')
            labels = labels.to('cuda')
            # Forward pass (the hook will capture fc1 activations)
            outputs = model(images)
            # Save the labels for each batch
            fc_labels.extend(labels.cpu().numpy())

    hook_handle.remove()

    # Concatenate all feature batches
    features_array = torch.cat(fc_features, dim=0).numpy()
    return features_array, fc_labels

def plot_samples_and_boundaries(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
    Z = model(torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to('cuda'))
    Z = torch.argmax(Z, dim=1).reshape(xx.shape).cpu().numpy()

    plt.figure(figsize=(6,5))
    plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.Spectral)
    for i in range(num_classes):
        plt.scatter(X[y==i, 0], X[y==i, 1], c=colors[i], label=f'Class {i}', alpha=0.7)
    plt.legend()
    plt.title(title)
    plt.xlabel("X1")
    plt.ylabel("X2")
    plt.show()

In [ ]:

print(f"Clean Accuracy: {validate(model, val_dataloader): .2%}")

with torch.no_grad():
    _, clean_features = model(x_val.to('cuda'), return_features=True)

reducer = visualize_features(clean_features.cpu(), y_val, "Clean Features")
plot_samples_and_boundaries(model, x_val, y_val, "Clean Data")

In [ ]:
x_val_noisy = add_gaussian_noise(x_val, severity=5)

plot_samples_and_boundaries(model, x_val_noisy, y_val, "Noisy Data")

noisy_test_loader = torch.utils.data.DataLoader(PointDataset(x_val_noisy, y_val),
                                                batch_size=64, shuffle=False)
with torch.no_grad():
    _, noisy_features = model(x_val_noisy.to('cuda'), return_features=True)

_ = visualize_features(noisy_features.cpu(), y_val, "Noisy Features", reducer)
zero_shot_accuracy = validate(model, noisy_test_loader)

In [ ]:
# 5. TENT adaptation: update only BatchNorm layers using entropy minimization
def entropy_loss(logits):
    return -(logits.softmax(1) * logits.log_softmax(1)).sum(1)

def collect_params(model, freeze_layers=[]):
    """Collect the affine scale + shift parameters from batch norms.
    Walk the model's modules and collect all batch normalization parameters.
    Return the parameters and their names.
    Note: other choices of parameterization are possible!
    """
    params = []
    names = []
    for nm, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm, nn.GroupNorm)):
            if any([f'{layer}' in nm for layer in freeze_layers]):
                continue
            for np, p in m.named_parameters():
                if np in ['weight', 'bias']:  # weight is scale, bias is shift
                    params.append(p)
                    names.append(f"{nm}.{np}")
    return params, names

def configure_model(model):
    """Configure model for use with eata."""
    # train mode, because eata optimizes the model to minimize entropy
    # self.model.train()
    model.eval()  # eval mode to avoid stochastic depth in swin. test-time normalization is still applied
    # disable grad, to (re-)enable only what eata updates
    model.requires_grad_(False)
    # configure norm for eata updates: enable grad + force batch statisics
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.requires_grad_(True)
            # force use of batch stats in train and eval modes
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
        elif isinstance(m, nn.BatchNorm1d):
            m.train()   # always forcing train mode in bn1d will cause problems for single sample tta
            m.requires_grad_(True)

            # # @TODO our addition
            # m.track_running_stats = False
            # m.running_mean = None
            # m.running_var = None
            
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
            m.requires_grad_(True)

configure_model(model)
bn_params, bn_names = collect_params(model, freeze_layers=["bn100"])
print("BatchNorm Parameters:", bn_names)
tent_optimizer = optim.Adam(bn_params, lr=1e-3)


In [ ]:
adaptation_data = x_val_noisy[y_val == 0] 
adaptation_labels = y_val[y_val == 0]
# adaptation_data = torch.cat((x_val_noisy[y_val == 0], x_val_noisy[y_val == 1]), dim=0) 
# adaptation_data = x_val_noisy
# Run a few adaptation steps on the noisy data


adaptation_loader = torch.utils.data.DataLoader(PointDataset(adaptation_data, adaptation_labels),
                                                batch_size=64, shuffle=True)

test_acc_list = []
logit_norm_list = []

test_acc_list.append((zero_shot_accuracy,0))


epochs = 10
steps = 0

total_steps = len(adaptation_loader) * epochs
log_frequency = 2

for epoch in range(epochs):

    for x_data, y_data in adaptation_loader:
        tent_optimizer.zero_grad()
        x_data, y_data = x_data.to('cuda'), y_data.to('cuda')
        logits = model(x_data)
        loss = entropy_loss(logits).mean(0)
        loss.backward()
        tent_optimizer.step()


        if (steps+1) % log_frequency == 0:
            # test accuracy
            test_accuracy = validate(deepcopy(model), noisy_test_loader)
            test_acc_list.append((test_accuracy, steps))
            print(f"Adaptation step {steps}/{total_steps}, Entropy Loss: {loss.item():.4f}")
            print(f"Test Accuracy: {test_accuracy:.2f}")

            
            logit_norm, _ = calc_logit_norm(torch.Tensor(logits[:,[0]]))
            logit_norm_list.append((logit_norm, steps))

        steps += 1

# plot test accuracy
plt.figure(figsize=(6,3))
step_list = [x[1] for x in test_acc_list]
accuracy_list = [x[0] for x in test_acc_list]
plt.plot(step_list, accuracy_list)
plt.xlabel("Adaptation Steps")
plt.ylabel("Test Accuracy")

plt.figure(figsize=(6,3))
logit_norms = [x[0] for x in logit_norm_list]
plt.plot(step_list[1:], logit_norms)
plt.xlabel("Adaptation Steps")
plt.ylabel("Logit Norm")

# Extract features after TENT adaptation
with torch.no_grad():
    _, noisy_features = model(x_val_noisy.to('cuda'), return_features=True)

_ = visualize_features(noisy_features.cpu(), y_val, "Features after Imbalanced Adaptation", reducer)
